<a href="https://colab.research.google.com/github/JakeOh/202511_BD53/blob/main/lab_ml/ml22_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXAONE 모델을 사용한 질문-답변 시스템

현재 Colab에 설치된 transformers 패키지의 버전은 5.0.0.

EXAONE 모델을 사용하기 위해서는 5.1.0 버전이 필요.

In [1]:
!pip install -U transformers==5.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 86.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Imports

In [2]:
import numpy as np
import scipy
import transformers

In [3]:
transformers.__version__

'5.1.0'

In [4]:
# 깃허브에 ipynb 파일을 업로드할 때 다운로드 상태(진행바) 표시줄 때문에 오류 발생
# -> 깃허브에 정상적으로 업로드되게 하기 위해서
transformers.utils.logging.disable_progress_bar()

# EXAONE-3.5 모델

In [6]:
pipe = transformers.pipeline(
    task='text-generation',
    model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',
    device=0,  # GPU 사용
    trust_remote_code=True
)

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [7]:
# 메시지 템플릿: 'role'과 'content'를 키로 갖는 dict들을 아이템으로 갖는 리스트.
# role: 역할(system, user, assistant)
# content: 내용
messages = [
    {
        'role': 'system',
        'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야. \
        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는 \
        간단하고 친절한 답변을 생성해줘.'
    },
    {
        'role': 'user',
        'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'
    }
]

In [16]:
result = pipe(messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
print(result)
#> pipeline 호출 결과: 'generated_text' 키를 갖는 dict 1개를 저장하고 있는 리스트

[{'generated_text': [{'role': 'system', 'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'}, {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'}, {'role': 'assistant', 'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]}]


In [18]:
len(result)

1

In [19]:
result[0]  #> dict

{'generated_text': [{'role': 'system',
   'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
  {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
  {'role': 'assistant',
   'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]}

In [20]:
result[0].keys()

dict_keys(['generated_text'])

result - `[ { 'generated_text': [...] } ]`

In [21]:
result[0]['generated_text']
#> 'role'과 'content'를 키로 갖는 dict들의 list.
#> 3개의 dict: 첫 2개는 입력값, 마지막 1개 AI가 생성한 답변

[{'role': 'system',
  'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
 {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
 {'role': 'assistant',
  'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]

In [22]:
result[0]['generated_text'][-1]

{'role': 'assistant',
 'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}

In [23]:
result[0]['generated_text'][-1]['content']

'안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'

pipeline은 실행할 때마다 다른 답변(텍스트)를 생성함.

pipeline의 파라미터:

*   `return_full_text`: 이전의 대화 기록을 모두 리턴할 지 여부를 설정. 기본값은 True.
*   `do_sample`: 샘플링 전략을 사용할 지 여부를 설정. 기본값은 True.
    *   `do_sample=True`: 메시지 프롬프트가 같아도 실행할 때마다 생성되는 텍스트가 달라짐.
    *   `do_sample=False`
        *   메시지 프롬프트가 같으면 항상 같은 텍스트가 생성됨.
        *   가장 확률이 높은 토큰들만 선택해서 텍스트를 생성.
*   샘플링 전략
    *   temperature(온도)
    *   top-k 방식
    *   top-p 방식


In [26]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              do_sample=False)
print(result)
#> result - [ { 'generated_text': '답변' } ]
#> 실행할 때마다 항상 같은 답변.

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금 더 자세히 알아보시려면 연락을 취해보시는 건 어떨까요? 친절하게 답변해 드리겠습니다! 😊'}]


# 샘플링 전략 - temperature